# Clean held-out test set

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../../"))
from src.config import BASE_PATH
from src.utils import col_config_og
import pandas as pd
import src.clean as clean
from shutil import rmtree

export_dir = BASE_PATH / "data/cleaned/processed/test"

## Clean

In [ ]:
df_test = pd.read_parquet(BASE_PATH / "data/cleaned/initial_split/test.parquet")
df_test.shape

In [ ]:
meta_cols = col_config_og["meta"]

self_impact_cols = col_config_og["self_impact"]
parent_impact_cols = col_config_og["parent_impact"]

self_text_cols = col_config_og["self_text"]
parent_text_cols = col_config_og["parent_text"]

self_quest_cols = col_config_og["self_quest"]
parent_quest_cols = col_config_og["parent_quest"]


all_self_cols = self_quest_cols + self_text_cols + self_impact_cols
all_parent_cols = parent_quest_cols + parent_text_cols + parent_impact_cols
all_depr_cols = all_self_cols + all_parent_cols

keep_cols = meta_cols + all_depr_cols
df_subset = df_test[keep_cols]
df_subset.shape

In [ ]:
self_df = df_subset[meta_cols + all_self_cols].copy()
parent_df = df_subset[meta_cols + all_parent_cols].copy()

# Create Depression/Irritability/Loss of Interest checks
self_df_w_cols = clean.create_checkpoint_cols(
    df=self_df, impact_type="self", impact_cols=self_impact_cols
)
parent_df_w_cols = clean.create_checkpoint_cols(
    df=parent_df, impact_type="parent", impact_cols=parent_impact_cols
)

# Fill missings w/ 0 when not prompted to answer questions, given answers to previous questions
self_df_coded = clean.mark_NA_0_when_not_prompted(
    df=self_df_w_cols, impact_type="self", impact_cols=self_impact_cols
)
parent_df_coded = clean.mark_NA_0_when_not_prompted(
    df=parent_df_w_cols, impact_type="parent", impact_cols=parent_impact_cols
)

# Drop healthy patients to avoid inflated results from easy-to-predict entries
self_df_filter = clean.remove_0_impact_participants(
    df=self_df_coded, impact_type="self", impact_cols=self_impact_cols
)
parent_df_filter = clean.remove_0_impact_participants(
    df=parent_df_coded, impact_type="parent", impact_cols=parent_impact_cols
)

# Clean up remaining NAs + drop heavily missing columns
self_df_final = clean.remove_remaining_missing(
    df=self_df_filter,
    impact_type="self",
    quest_cols=self_quest_cols,
    text_cols=self_text_cols,
)
parent_df_final = clean.remove_remaining_missing(
    df=parent_df_filter,
    impact_type="parent",
    quest_cols=parent_quest_cols,
    text_cols=parent_text_cols,
)

Export

In [ ]:
# if export_dir.exists():
#     rmtree(export_dir)
# export_dir.mkdir(exist_ok=False, parents=True)
# # Self
# self_df_final = self_df_final.reset_index(drop=True)
# self_df_final.to_parquet(export_dir / "self.parquet")
# # Parent
# parent_df_final = parent_df_final.reset_index(drop=True)
# parent_df_final.to_parquet(export_dir / "parent.parquet")

## Format text

In [ ]:
from src.embed import concat_question_answer

## Meta data
meta_dir = BASE_PATH / "data" / "cleaned" / "meta_text"
meta_self = pd.read_csv(meta_dir / "self.csv", index_col=0)
meta_parent = pd.read_csv(meta_dir / "parent.csv", index_col=0)

self_df_final = pd.read_parquet(export_dir / "self.parquet")
parent_df_final = pd.read_parquet(export_dir / "parent.parquet")

In [ ]:
qa_concat_self = concat_question_answer(answer_df=self_df_final, question_df=meta_self)
qa_concat_parent = concat_question_answer(
    answer_df=parent_df_final, question_df=meta_parent
)

In [ ]:
# export_dir_qa = BASE_PATH / "data/cleaned/question_answer/test"
# if export_dir_qa.exists():
#     rmtree(export_dir_qa)
# export_dir_qa.mkdir(exist_ok=False, parents=True)
# # Self
# qa_concat_self = qa_concat_self.reset_index(drop=True)
# qa_concat_self.to_parquet(export_dir_qa / "self.parquet")
# # Parent
# qa_concat_parent = qa_concat_parent.reset_index(drop=True)
# qa_concat_parent.to_parquet(export_dir_qa / "parent.parquet")